# Imports

In [2]:
import requests
from pathlib import Path
import os
from math import radians, cos, sin, asin, sqrt
from itertools import permutations
import time
import pandas as pd
from datetime import datetime, timedelta

# Definition of constants, API keys, helper function

In [3]:
BASE_DB_API = "https://apis.deutschebahn.com/db-api-marketplace/apis/"
STOP_PLACES_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stop-places"
RIS_STATION_URL = BASE_DB_API + "ris-station/v1/stations/"

V6_TRANSPORT_BASE_URL = "https://v6.db.transport.rest"

ROOT_DIR = Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = ROOT_DIR / "Raw_Data"
PROCESSED_DATA_DIR = ROOT_DIR / "Processed_Data"

scraping_raw_output_file = RAW_DATA_DIR / "train_routes_scraped.csv"
scraping_cleaned_output_file = PROCESSED_DATA_DIR / "train_routes_scraped.csv"

DB_CLIENT_ID = "a9f83c55d26c3ee7f48f4ce887ec2a57"
DB_API_KEY = "422cac21a0a83876c75efb8806589ea0"

header_ris = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/vnd.de.db.ris+json"
}

SCRAPING_RATE_LIMIT_SLEEP = 1.0  # Seconds to sleep between requests to respect 100 req/min

def haversine(lat1, lon1, lat2, lon2):
    """ Calculate distance in km between two points """
    R = 6371 # Earth radius
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    a = sin(dLat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dLon/2)**2
    return 2 * R * asin(sqrt(a))

#  Railway stations selection & filtering

First, match train station with selected cities from simplemaps ( Processed_Data/cities_500K). Later for matched cities retrieve required timetables.

Matching should be done geospatially

In [4]:
cities_data = pd.read_csv(PROCESSED_DATA_DIR / "cities_500K.csv")
cities_data.rename(columns={"name_city":"name", "lat_city":"latitude", "lng_city":"longitude","country":"iso_code"}, inplace=True)
cities_data.head()

,name,latitude,longitude,iso_code,population,is_capital
0,Vienna,48.2083,16.3725,AT,1973403.0,True
1,Brussels,50.8467,4.3525,BE,1235192.0,True
2,Antwerp,51.2178,4.4003,BE,536079.0,False
3,Sofia,42.7000,23.3300,BG,1383435.0,True
4,Prague,50.0875,14.4214,CZ,1357326.0,True


# Define functions to filter cities

In [5]:
def get_stations_api_stop_places(cities, base_url = STOP_PLACES_URL, limit = 3):
    """
    Retrieves station information from the Deutsche Bahn API based on the station name.

    Keyword arguments:
    cities -- DataFrame containing city information
    base_url -- Base URL for the API
    limit -- Maximum number of stations to retrieve per city
    Return: DataFrame of stations for given cities
    """

    retrieved_stations = pd.DataFrame(columns=['city_name', 'eva_id', 'station_name', 'latitude', 'longitude'])

    for _, row in cities.iterrows():
        city_name = row['name']
        response = None
        params = {
            "sortBy": "RELEVANCE",
            "onlyActive": "true",
            "withSynonyms": "true",
            "latitude": row['latitude'],
            "longitude": row['longitude'],
            "limit": limit
        }
        success = False
        retries = 0

        while not success and retries < 3:
            response = requests.get(url=f"{base_url}/by-name/{city_name}", params=params, headers=header_ris)

            if response.status_code == 429:
                print(f"Rate limit reached. Sleeping for 1s...")
                time.sleep(1)
                retries += 1
                continue

            if response.status_code == 200:
                stations = response.json().get('stopPlaces', [])
                print(f"City: {city_name}, Stations Found: {len(stations)}")
                for station in stations:
                    latitude = float(station.get('position').get('latitude'))
                    longitude = float(station.get('position').get('longitude'))

                    if haversine(row['latitude'], row['longitude'], latitude, longitude) > 10:
                        print(f"Skipping station {station.get('names').get('DE').get('nameLong')} due to distance.")
                        continue

                    retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({
                        'city_name': [city_name],
                        'eva_id': [str(station.get('evaNumber'))],
                        'station_name': [str(station.get('names').get('DE').get('nameLong'))],
                        'latitude': [float(station.get('position').get('latitude'))],
                        'longitude': [float(station.get('position').get('longitude'))]
                    })])
                success = True
            else:
                print(f"Error retrieving stations for city {city_name}: {response.status_code}")
                break

            time.sleep(0.09)  # To respect API rate limits

    return retrieved_stations

# Get first station for every city

In [6]:
cities_stations = get_stations_api_stop_places(cities_data, limit=1)

City: Vienna, Stations Found: 1


/var/folders/pj/dyqj7jwx6r76zl2c1w7dvc7h0000gn/T/ipykernel_17296/1597653662.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Brussels, Stations Found: 1
City: Antwerp, Stations Found: 1
City: Sofia, Stations Found: 1
Skipping station Sofie-Hammer-Straße, Osnabrück due to distance.
City: Prague, Stations Found: 1
City: Berlin, Stations Found: 1
City: Stuttgart, Stations Found: 1
City: Munich, Stations Found: 1
City: Hamburg, Stations Found: 1
City: Cologne, Stations Found: 1
City: Frankfurt, Stations Found: 1
City: Düsseldorf, Stations Found: 1
City: Leipzig, Stations Found: 1
City: Dortmund, Stations Found: 1
City: Essen, Stations Found: 1
City: Bremen, Stations Found: 1
City: Dresden, Stations Found: 1
City: Hannover, Stations Found: 1
City: Nuremberg, Stations Found: 1
City: Duisburg, Stations Found: 1
City: Copenhagen, Stations Found: 1
City: Tallinn, Stations Found: 1
Skipping station Tallinner Straße, Schwerin (Meckl) due to distance.
City: Madrid, Stations Found: 1
Skipping station Madrider Ring, Würzburg due to distance.
City: Barcelona, Stations Found: 1
City: Valencia, Stations Found: 1
Skippi

# Inspect for which cities stations have and have not been found

## No stations found

In [7]:
cities_no_station = pd.merge(cities_data, cities_stations, how='outer',left_on=['name'], right_on=['city_name'], indicator=True).query('_merge == "left_only"')
cities_no_station.reset_index(inplace=True)
cities_no_station

,index,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y,_merge
0,2,Athens,37.9842,23.7281,GR,643452.0,True,NaN,NaN,NaN,NaN,NaN,left_only
1,9,Bucharest,44.4325,26.1039,RO,1716961.0,True,NaN,NaN,NaN,NaN,NaN,left_only
2,20,Gothenburg,57.7075,11.9675,SE,607882.0,False,NaN,NaN,NaN,NaN,NaN,left_only
3,23,Helsinki,60.1708,24.9375,FI,664921.0,True,NaN,NaN,NaN,NaN,NaN,left_only
4,26,Lisbon,38.7253,-9.1500,PT,548703.0,True,NaN,NaN,NaN,NaN,NaN,left_only
5,30,Madrid,40.4169,-3.7033,ES,3266126.0,True,NaN,NaN,NaN,NaN,NaN,left_only
6,34,Málaga,36.7194,-4.4200,ES,586384.0,False,NaN,NaN,NaN,NaN,NaN,left_only
7,35,Naples,40.8333,14.2500,IT,913462.0,False,NaN,NaN,NaN,NaN,NaN,left_only
8,40,Riga,56.9489,24.1064,LV,660187.0,True,NaN,NaN,NaN,NaN,NaN,left_only
9,43,Sevilla,37.3900,-5.9900,ES,684025.0,False,NaN,NaN,NaN,NaN,NaN,left_only


## Stations found

In [8]:
cities_with_stations = pd.merge(cities_data, cities_stations, how='inner',left_on=['name'], right_on=['city_name'])
cities_with_stations

,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y
0,Vienna,48.2083,16.3725,AT,1973403.0,True,Vienna,8103000,Wien Hbf,48.185101,16.377113
1,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
2,Antwerp,51.2178,4.4003,BE,536079.0,False,Antwerp,8800007,Antwerpen Centraal,51.215811,4.421168
3,Prague,50.0875,14.4214,CZ,1357326.0,True,Prague,5400014,Praha hl.n.,50.083062,14.436039
4,Berlin,52.5200,13.4050,DE,3755251.0,True,Berlin,8011160,Berlin Hbf,52.525592,13.369545
5,Stuttgart,48.7775,9.1800,DE,632865.0,False,Stuttgart,8000096,Stuttgart Hbf,48.784780,9.182757
6,Munich,48.1375,11.5750,DE,1512491.0,False,Munich,8000261,München Hbf,48.140232,11.558335
7,Hamburg,53.5500,10.0000,DE,1892122.0,False,Hamburg,8002549,Hamburg Hbf,53.552736,10.006909
8,Cologne,50.9364,6.9528,DE,1084831.0,False,Cologne,8003368,Köln Messe/Deutz,50.940874,6.975001
9,Frankfurt,50.1106,8.6822,DE,773068.0,False,Frankfurt,8000105,Frankfurt(Main)Hbf,50.106682,8.662828


# Definition of necessary functions for route scraping

In [9]:
def get_next_representative_weekday():
    """
    Finds the next Tuesday or Wednesday.
    Mid-week days have the most consistent 'standard' schedules.
    """
    now = datetime.now()
    # 0=Mon, 1=Tue, 2=Wed...
    days_ahead = (1 - now.weekday() + 7) % 7 # Target Tuesday
    if days_ahead == 0: days_ahead = 7 # If today is Tuesday, get next week

    target_date = now + timedelta(days=days_ahead)
    # set to 08:00 AM - Peak morning traffic usually has the best connections
    return target_date.replace(hour=8, minute=0, second=0, microsecond=0)

def fetch_fastest_connection(origin_id, dest_id):
    """
    Makes a SINGLE smart request to find the fastest connection.
    Retrieves 5 options and picks the minimum duration.
    """
    target_date = get_next_representative_weekday()

    params = {
        "from": origin_id,
        "to": dest_id,
        "departure": target_date.isoformat(),
        "results": 5,           # Get 5 options in ONE request
        "national": "true",     # Prefer High Speed
        "nationalExpress": "true",
        "transfers": 4          # Allow complex routes if they are faster
    }

    # Retry logic for stability
    for attempt in range(3):
        try:
            response = requests.get(f"{V6_TRANSPORT_BASE_URL}/journeys", params=params, timeout=10)

            if response.status_code == 200:
                data = response.json()
                journeys = data.get('journeys', [])

                if not journeys:
                    return None

                # Optimization: Process all 5 results locally to find the minimum
                min_duration = float('inf')
                best_journey = None

                for journey in journeys:
                    if not journey.get('legs'): continue

                    dep = datetime.fromisoformat(journey['legs'][0]['departure'])
                    arr = datetime.fromisoformat(journey['legs'][-1]['arrival'])
                    duration = (arr - dep).total_seconds() / 60

                    if duration < min_duration:
                        min_duration = duration
                        best_journey = journey
                        best_journey['calculated_duration'] = int(duration)

                # Extract details from the winner
                legs = best_journey['legs']
                train_names = [l.get('line', {}).get('name', '') for l in legs if l.get('mode') == 'train']

                time.sleep(SCRAPING_RATE_LIMIT_SLEEP) # Respect limits
                return {
                    "origin_id": origin_id,
                    "destination_id": dest_id,
                    "origin_name": legs[0]['origin']['name'],
                    "destination_name": legs[-1]['destination']['name'],
                    "min_duration_minutes": best_journey['calculated_duration'],
                    "transfers": len(legs) - 1,
                    "trains": ", ".join(filter(None, train_names)),
                    "check_date": target_date.strftime("%Y-%m-%d")
                }

            elif response.status_code == 429:
                time.sleep(3 * (attempt + 1)) # Backoff
            elif response.status_code >= 500:
                time.sleep(2)
            else:
                return None

        except Exception as e:
            time.sleep(1)

    return None

def get_efficient_network_data(eva_ids):
    # use permutations because A->B might be slightly different than B->A
    # (e.g. connections matching up)
    pairs = list(permutations(eva_ids, 2))

    results = []

    for i, (origin, dest) in enumerate(pairs):
        print(f"[{i+1}/{len(pairs)}] {origin} -> {dest}...", end=" ", flush=True)

        data = fetch_fastest_connection(origin, dest)

        if data:
            print(f"found: {data['min_duration_minutes']} min")
            results.append(data)
        else:
            print(f"no route found")

    return pd.DataFrame(results)

# Define function to Scrape API with intermittent saves to prevent data loss

In [10]:
def scrape_train_routes_with_checkpoints(eva_ids, batch_size = 10):
    """
    scrapes train routes and persists to CSV every batch_size API calls.

    Args:
        eva_ids: List of station EVA IDs to query
        batch_size: Save to CSV after this many successful API calls (default: 10)
    """

    processed_pairs = set()
    if scraping_raw_output_file.exists():
        existing_df = pd.read_csv(scraping_raw_output_file)
        processed_pairs = set(zip(existing_df['origin_id'], existing_df['destination_id']))
        print(f"found {len(processed_pairs)} existing routes, resuming scraping..")
    else:
        print(f"no existing routes found, created new output file {scraping_raw_output_file}")

    pairs = list(permutations(eva_ids, 2))
    pending_pairs = [p for p in pairs if p not in processed_pairs]

    batch_results = []
    api_calls_count = 0
    failed_count = 0

    try:
        for idx, (origin, dest) in enumerate(pending_pairs):
            pair_num = idx + len(processed_pairs) + 1
            print(f"[{pair_num}/{len(pairs)}] {origin} → {dest}...", end=" ", flush=True)

            try:
                start_time = time.time()
                data = fetch_fastest_connection(origin, dest)
                elapsed = time.time() - start_time

                if data:
                    print(f"{data['min_duration_minutes']} min (retrieved in {elapsed} seconds)")
                    batch_results.append(data)
                    api_calls_count += 1
                else:
                    print(f"no route found (retrieved in {elapsed} seconds)")

                    # if no route was found, still add it to output dataset.
                    # this is so that the scraping does not try again for this city combination when restarted
                    data = {
                        "origin_id": origin,
                        "destination_id": dest,
                        "origin_name": "",
                        "destination_name": "",
                        "min_duration_minutes": -1,
                        "transfers": 0,
                        "trains": "",
                        "check_date": get_next_representative_weekday().strftime("%Y-%m-%d")
                    }
                    batch_results.append(data)

                    api_calls_count += 1
                    failed_count += 1

                # batch save every N successful calls
                if api_calls_count >= batch_size:
                    batch_df = pd.DataFrame(batch_results)

                    # append to CSV or create if doesn't exist yet
                    if scraping_raw_output_file.exists():
                        batch_df.to_csv(scraping_raw_output_file, mode='a', header=False, index=False)
                    else:
                        batch_df.to_csv(scraping_raw_output_file, mode='w', header=True, index=False)

                    print(f"\nintermittent save: saved {api_calls_count} routes to {scraping_raw_output_file.name}")

                    batch_results = []
                    api_calls_count = 0

            except Exception as e:
                print(f"error: {e}")
                failed_count += 1
                time.sleep(2) # let's chill a bit after error to be safe
                continue

        # final save of remaining results
        if batch_results:
            batch_df = pd.DataFrame(batch_results)
            if scraping_raw_output_file.exists():
                batch_df.to_csv(scraping_raw_output_file, mode='a', header=False, index=False)
            else:
                batch_df.to_csv(scraping_raw_output_file, mode='w', header=True, index=False)
            print(f"\nsaved {len(batch_results)} remaining routes")

        print(f"scraping complete!")

    except KeyboardInterrupt:
        print(f"interrupted by user")
        # save any remaining batch data before exiting
        if batch_results:
            batch_df = pd.DataFrame(batch_results)
            if scraping_raw_output_file.exists():
                batch_df.to_csv(scraping_raw_output_file, mode='a', header=False, index=False)
            else:
                batch_df.to_csv(scraping_raw_output_file, mode='w', header=True, index=False)
            print(f"saved {len(batch_results)} routes before exit")
        raise

# Run scraping with intermittent saves

Note: as this notebook was refactored as not to include unnecessary code, the full output is not included here.
Overall scraping took approximately 3 hours.

In [11]:
scrape_train_routes_with_checkpoints(cities_stations['eva_id'].tolist(), batch_size=10)

no existing routes found, created new output file /Users/rafael/projects/DOPP_groupB_2025W/Raw_Data/train_routes_scraped2.csv
[1/1806] 8103000 → 8800004... 622 min (retrieved in 1.2305307388305664 seconds)
[2/1806] 8103000 → 8800007... 653 min (retrieved in 1.2207386493682861 seconds)
[3/1806] 8103000 → 5400014... 253 min (retrieved in 1.2341680526733398 seconds)
[4/1806] 8103000 → 8011160... 429 min (retrieved in 2.320432186126709 seconds)
[5/1806] 8103000 → 8000096... 391 min (retrieved in 2.121380090713501 seconds)
[6/1806] 8103000 → 8000261... 253 min (retrieved in 1.9782400131225586 seconds)
[7/1806] 8103000 → 8002549... 524 min (retrieved in 2.882812976837158 seconds)
[8/1806] 8103000 → 8003368... 481 min (retrieved in 3.008584976196289 seconds)
[9/1806] 8103000 → 8000105... 384 min (retrieved in 2.169938802719116 seconds)
[10/1806] 8103000 → 8000085... 499 min (retrieved in 2.830449104309082 seconds)

intermittent save: saved 10 routes to train_routes_scraped2.csv
[11/1806] 8103

KeyboardInterrupt: 

# Remove duplicates from saved data

In [12]:
df = pd.read_csv(scraping_raw_output_file)

df = df.drop_duplicates()

df.to_csv(scraping_cleaned_output_file, index=False)

# Add back cities to routes data

Match up train stations with city names again, for easier data matching down the line

In [13]:
df_routes = pd.read_csv(scraping_cleaned_output_file)

df_routes.rename(columns={"origin_name": "origin_station_name", "destination_name": "destination_station_name"}, inplace=True)

cities_stations_names_only = cities_stations[["city_name", "station_name"]]

cities_stations_origin = cities_stations.add_prefix("origin_")
cities_stations_destination = cities_stations.add_prefix("destination_")

df_routes_with_city_names = df_routes.merge(cities_stations_origin, on="origin_station_name")
df_routes_with_city_names = df_routes_with_city_names.merge(cities_stations_destination, on="destination_station_name")

df_routes_with_city_names.to_csv(scraping_cleaned_output_file, index=False)

df_routes_with_city_names

,origin_id,destination_id,origin_station_name,destination_station_name,min_duration_minutes,transfers,trains,check_date,origin_city_name,origin_eva_id,origin_latitude,origin_longitude,destination_city_name,destination_eva_id,destination_latitude,destination_longitude
0,8103000,8800004,Wien Hbf,Bruxelles Midi,622,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Brussels,8800004,50.835376,4.335694
1,8103000,8800007,Wien Hbf,Antwerpen Centraal,653,4,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Antwerp,8800007,51.215811,4.421168
2,8103000,5400014,Wien Hbf,Praha hl.n.,253,0,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Prague,5400014,50.083062,14.436039
3,8103000,8011160,Wien Hbf,Berlin Hbf,429,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Berlin,8011160,52.525592,13.369545
4,8103000,8000096,Wien Hbf,Stuttgart Hbf,391,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Stuttgart,8000096,48.784780,9.182757
5,8103000,8000261,Wien Hbf,München Hbf,253,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Munich,8000261,48.140232,11.558335
6,8103000,8002549,Wien Hbf,Hamburg Hbf,524,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Hamburg,8002549,53.552736,10.006909
7,8103000,8000105,Wien Hbf,Frankfurt(Main)Hbf,384,0,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Frankfurt,8000105,50.106682,8.662828
8,8103000,8000085,Wien Hbf,Düsseldorf Hbf,499,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Düsseldorf,8000085,51.219962,6.794319
9,8103000,8010205,Wien Hbf,Leipzig Hbf,378,4,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Leipzig,8010205,51.345471,12.382064
